# Analisis Pola Kenaikan dan Penurunan Harga Saham

Notebook ini berfokus pada **pola pergerakan harga saham saja**. Tidak menggunakan foto, gambar, atau dataset gambar. Visualisasi yang ditampilkan berupa grafik garis dari harga saham sepanjang waktu.

Tujuan analisis:
- melihat pola harga saham naik dan turun;
- menghitung perubahan harga harian dalam persentase;
- mengelompokkan perubahan menjadi **Naik**, **Turun**, atau **Tetap**;
- membuat grafik harga dan grafik perubahan harian.

Gunakan dataset CSV historis saham dengan kolom `Date` dan `Close`.

In [ ]:
# 1. Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

# Upload dataset saham CSV
uploaded = files.upload()

In [ ]:
# 2. Membaca dataset
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)
df.columns = df.columns.str.strip()

print("Ukuran dataset:", df.shape)
print("Kolom dataset:", df.columns.tolist())
display(df.head())

In [ ]:
# 3. Membersihkan data
if 'Date' not in df.columns or 'Close' not in df.columns:
    raise ValueError("CSV wajib memiliki kolom Date dan Close.")

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Close'] = pd.to_numeric(
    df['Close'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

df = (df.dropna(subset=['Date', 'Close'])
        .drop_duplicates()
        .sort_values('Date')
        .reset_index(drop=True))

print("Jumlah data setelah dibersihkan:", len(df))
display(df[['Date', 'Close']].head())

In [ ]:
# 4. Menghitung perubahan harga dan pola pergerakan
df['Perubahan_Harga'] = df['Close'].diff()
df['Perubahan_Persen'] = df['Close'].pct_change() * 100

def tentukan_pola(perubahan):
    if pd.isna(perubahan):
        return 'Belum ada pembanding'
    elif perubahan > 0:
        return 'Naik'
    elif perubahan < 0:
        return 'Turun'
    return 'Tetap'

df['Pola'] = df['Perubahan_Harga'].apply(tentukan_pola)
display(df[['Date', 'Close', 'Perubahan_Harga', 'Perubahan_Persen', 'Pola']].head(10))

In [ ]:
# 5. Grafik garis pola harga saham
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Close'], linewidth=1.8)
plt.title('Pola Kenaikan dan Penurunan Harga Saham')
plt.xlabel('Tanggal')
plt.ylabel('Harga Penutupan (Close)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Grafik perubahan persentase harian
data_perubahan = df.dropna(subset=['Perubahan_Persen'])

plt.figure(figsize=(12, 5))
plt.plot(data_perubahan['Date'], data_perubahan['Perubahan_Persen'], linewidth=1.2)
plt.axhline(0, linewidth=1)
plt.title('Perubahan Harga Saham Harian (%)')
plt.xlabel('Tanggal')
plt.ylabel('Perubahan (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Ringkasan jumlah pola naik, turun, dan tetap
ringkasan = df.loc[df['Pola'].isin(['Naik', 'Turun', 'Tetap']), 'Pola'].value_counts()
print("Jumlah hari berdasarkan pola:")
display(ringkasan.rename_axis('Pola').to_frame('Jumlah Hari'))

plt.figure(figsize=(7, 4))
ringkasan.reindex(['Naik', 'Turun', 'Tetap'], fill_value=0).plot(kind='bar')
plt.title('Frekuensi Pola Pergerakan Harga Saham')
plt.xlabel('Pola')
plt.ylabel('Jumlah Hari')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Ringkasan statistik
print("Periode data:", df['Date'].min().date(), "sampai", df['Date'].max().date())
print(f"Harga penutupan terendah: {df['Close'].min():,.2f}")
print(f"Harga penutupan tertinggi: {df['Close'].max():,.2f}")
print(f"Harga penutupan awal: {df['Close'].iloc[0]:,.2f}")
print(f"Harga penutupan akhir: {df['Close'].iloc[-1]:,.2f}")

perubahan_total = (df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100
print(f"Perubahan total selama periode: {perubahan_total:.2f}%")

if perubahan_total > 0:
    print("Kesimpulan awal: harga akhir lebih tinggi daripada harga awal.")
elif perubahan_total < 0:
    print("Kesimpulan awal: harga akhir lebih rendah daripada harga awal.")
else:
    print("Kesimpulan awal: harga akhir sama dengan harga awal.")

## Kesimpulan

Analisis ini menampilkan pola kenaikan dan penurunan harga saham berdasarkan kolom harga penutupan (`Close`) pada dataset. Grafik pertama memperlihatkan arah pergerakan harga dari waktu ke waktu, sedangkan grafik kedua memperlihatkan persentase perubahan harian. Ringkasan frekuensi menunjukkan berapa kali harga tercatat naik, turun, atau tetap dibandingkan data sebelumnya.

Hasil kesimpulan bergantung pada CSV yang digunakan. Pola historis tidak menjamin pola yang sama akan terjadi di masa depan dan bukan merupakan saran investasi.